**Raspakivanje zip foldera**

In [ ]:
import os

zip_file_name = '' # naziv zip file-a
unzipped_dir_name = '' # naziv unzipped file-a

if not os.path.exists(unzipped_dir_name):
    print(f"Raspakujem {zip_file_name}...")
    !unzip -q {zip_file_name}
    print(f"'{zip_file_name}' uspešno raspakovan u '{unzipped_dir_name}'.")
else:
    print(f"Folder '{unzipped_dir_name}' već postoji. Preskačem raspakivanje.")

**Učitavanje i parsiranje labela**

In [ ]:
import os

BASE_DIR = " " # naziv direktorijuma ili baze

def load_dataset(split):
    images_dir = os.path.join(BASE_DIR, "images", split)
    labels_dir = os.path.join(BASE_DIR, "labels", split)

    data = []

    for img_name in os.listdir(images_dir):
        if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        base = os.path.splitext(img_name)[0]
        label_path = os.path.join(labels_dir, base + ".txt")

        if not os.path.exists(label_path):
            continue

        with open(label_path) as f:
            parts = f.read().strip().split()

        if len(parts) != 3:
            continue

        class_id = int(parts[0])
        start_x = float(parts[1])
        end_x = float(parts[2])

        data.append({
            "image_path": os.path.join(images_dir, img_name),
            "class_id": class_id,
            "start_x": start_x,
            "end_x": end_x
        })

    return data


train_data = load_dataset("train")
val_data   = load_dataset("val")
test_data = load_dataset("test")

print(f"Train samples: {len(train_data)}")
print(f"Val samples:   {len(val_data)}")
print("Test samples:", len(test_data))


**Kreiranje klase data generator**

In [ ]:
import numpy as np
import tensorflow as tf
from PIL import Image
import math

BATCH_SIZE = 32

class DataGenerator(tf.keras.utils.Sequence):

    def __init__(self, data, shuffle=True):
        self.data = data
        self.shuffle = shuffle
        self.indexes = np.arange(len(data))

        img = Image.open(data[0]["image_path"]).convert("RGB")
        self.H, self.W = img.size[1], img.size[0]

        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.data) / BATCH_SIZE)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * BATCH_SIZE:(idx + 1) * BATCH_SIZE]
        batch = [self.data[i] for i in batch_idx]

        X = np.zeros((len(batch), self.H, self.W, 3), dtype=np.float32)
        y_class = np.zeros((len(batch),), dtype=np.int32)
        y_start = np.zeros((len(batch), 1), dtype=np.float32)
        y_end   = np.zeros((len(batch), 1), dtype=np.float32)

        valid = 0

        for item in batch:
            img = np.array(
                Image.open(item["image_path"]).convert("RGB"),
                dtype=np.float32
            ) / 255.0

            start = float(item["start_x"])
            end   = float(item["end_x"])


            raw_cls = int(item["class_id"])

            if raw_cls == -1:
                cls = 2
                start, end = 0.0, 0.0
            else:
                cls = raw_cls

                if not (0.0 <= start <= 1.0 and 0.0 <= end <= 1.0):
                    continue
                if start == end:
                    continue
                if start > end:
                    start, end = end, start


            X[valid] = img
            y_class[valid] = cls
            y_start[valid, 0] = start
            y_end[valid, 0]   = end
            valid += 1

        return X[:valid], {
            "class_output": y_class[:valid],
            "start_output": y_start[:valid],
            "end_output": y_end[:valid]
            }

**Kreiranje objekata klase data generator**

In [ ]:
train_gen = DataGenerator(train_data, shuffle=True)
val_gen   = DataGenerator(val_data, shuffle=False)
test_gen  = DataGenerator(test_data, shuffle=False)
print(test_gen.data[1])

**Broj klasa za treniranje mreže**

In [ ]:
from collections import Counter

all_labels = []
for d in train_data:
    all_labels.append(d["class_id"])

print("Originalne klase u train_data:", Counter(all_labels))

**Broj klasa za testiranje mreže**

In [ ]:
from collections import Counter

all_labels = []
for d in test_data:
    all_labels.append(d["class_id"])

print("Originalne klase u test_data:", Counter(all_labels))

**Kreiranje funkcije za prikazivanje RGB vrednosti slike**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

def visualize_sample(sample):
    img = np.array(
        Image.open(sample["image_path"]).convert("RGB"),
        dtype=np.float32
    ) / 255.0

    H, W, _ = img.shape
    h = H // 2

    signal = img[h-1:h+2].mean(axis=0)   # (W, 3)

    start = int(sample["start_x"])
    end   = int(sample["end_x"])

    if start > end:
        start, end = end, start

    fig = plt.figure(figsize=(16, 10))

    ax1 = plt.subplot(3, 1, 1)
    ax1.imshow(img)
    ax1.axhline(h, color="yellow", linestyle="--", label="centralna linija")
    ax1.axvline(start, color="lime", linestyle="--", label="start_x")
    ax1.axvline(end, color="red", linestyle="--", label="end_x")
    ax1.set_title("Originalna slika")
    ax1.legend()

    ax2 = plt.subplot(3, 1, 2)
    ax2.plot(signal[:,0], label="R", color="r")
    ax2.plot(signal[:,1], label="G", color="g")
    ax2.plot(signal[:,2], label="B", color="b")
    ax2.axvline(start, color="lime", linestyle="--")
    ax2.axvline(end, color="red", linestyle="--")
    ax2.set_title("2D RGB signal")
    ax2.set_ylabel("Intenzitet")
    ax2.legend()

    ax3 = plt.subplot(3, 1, 3)
    energy = signal.mean(axis=1)
    ax3.plot(energy, color="black")
    ax3.axvline(start, color="lime", linestyle="--")
    ax3.axvline(end, color="red", linestyle="--")
    ax3.set_title("Intenzitet signala")
    ax3.set_xlabel("X pozicija")
    ax3.set_ylabel("Srednja vrednost RGB")

    plt.tight_layout()
    plt.show()


**Pozivanje funkcije za random uzorak**

In [ ]:
import random

sample = random.choice(train_data)
visualize_sample(sample)

**CNN model**

In [ ]:
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    GlobalAveragePooling2D, Dense
)
from tensorflow.keras.models import Model

NUM_CLASSES = 3
HEIGHT = train_gen.H
WIDTH  = train_gen.W

inputs = Input(shape=(None, WIDTH, 3))

x = Conv2D(32, (3,5), activation="relu", padding="same")(inputs)
x = MaxPooling2D((1,2))(x)

x = Conv2D(64, (3,5), activation="relu", padding="same")(x)
x = MaxPooling2D((1,2))(x)

x = Conv2D(128, (1,3), activation="relu", padding="same")(x)
x = GlobalAveragePooling2D()(x)

x = Dense(64, activation="relu")(x)

class_output = Dense(NUM_CLASSES, activation="softmax", name="class_output")(x)
start_output = Dense(1, activation="sigmoid", name="start_output")(x)
end_output   = Dense(1, activation="sigmoid", name="end_output")(x)

model = Model(inputs, [class_output, start_output, end_output])
model.summary()


**Funkcija za maskiranje**

In [ ]:
import tensorflow as tf

def masked_mse(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, 0.0), tf.float32)
    return tf.reduce_sum(mask * tf.square(y_true - y_pred)) / (tf.reduce_sum(mask) + 1e-6)

**Kompajliranje modela**

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(5e-4),
    loss={
        "class_output": "sparse_categorical_crossentropy",
        "start_output": masked_mse,
        "end_output": masked_mse
    },
    loss_weights={
        "class_output": 1.0,
        "start_output": 5.0,
        "end_output": 5.0
    },
    metrics={
        "class_output": "accuracy"
    }
)


**Dimenzije uzoraka za trening**

In [ ]:
x, y = train_gen[0]

print("X:", x.shape, x.min(), x.max())
for k, v in y.items():
    print(k, v.shape, "NaN:", np.isnan(v).any())


**Broj uzoraka u batch-u**

In [ ]:
from collections import Counter

X, y = train_gen[0]
print("Batch distribucija:", Counter(y["class_output"]))

**Treniranje modela**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=120,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

**Crtanje grafika za train loss za regresioni i klasifikacioni problem**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(history.history['loss'], label='Train loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation loss', linewidth=2)

best_epoch = min(
    range(len(history.history['val_loss'])),
    key=lambda i: history.history['val_loss'][i]
)
best_val = history.history['val_loss'][best_epoch]

plt.scatter(best_epoch, best_val, s=50)
plt.text(
    best_epoch,
    best_val,
    "",
    verticalalignment='bottom'
)

plt.xlabel("Epoch (trening iteracije)")
plt.ylabel("Ukupna funkcija greške")
plt.title("Konvergencija modela tokom treninga")

plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.plot(history.history['class_output_loss'], label='Trening')
plt.plot(history.history['val_class_output_loss'], label='Validacija')

plt.xlabel("Epoch (trening iteracije)")
plt.ylabel("Gubitak klasifikacije")
plt.title("Gubitak klasifikacije ambalаže")

plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(8,5))
plt.plot(history.history['start_output_loss'], label='Trening')
plt.plot(history.history['val_start_output_loss'], label='Validacija')

plt.xlabel("Epoch (trening iteracije)")
plt.ylabel("MSE")
plt.title("Srednja kvadratna greška početne X pozicije")

plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['end_output_loss'], label='Trening')
plt.plot(history.history['val_end_output_loss'], label='Validacija')

plt.xlabel("Epoch (trening iteracije)")
plt.ylabel("MSE")
plt.title("Srednja kvadratna greška krajnje X pozicije")

plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

**Matrica konfuzije na test skupu podataka**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
y_true, y_pred = [], []

for i in range(len(test_gen)):
    X, y = test_gen[i]
    preds = model.predict(X, verbose=0)
    y_pred.extend(np.argmax(preds[0], axis=1))
    y_true.extend(y["class_output"])

labels = [0, 1, 2]
names  = ["Limenka", "PET", "Neklasifikovano"]

cm = confusion_matrix(y_true, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(cm, display_labels=names)
disp.plot(cmap="Blues")

plt.xlabel("Predviđena klasa")
plt.ylabel("Stvarna klasa")
plt.title("")
plt.show()

**Metrike evaluacije sistema (Accuracy, Precision, Recall, F1 Score)**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true, y_pred,
    target_names=names
))

In [ ]:
model.save(" ") # čuvanje modela